# Data preprocessing

`data/pipeline.py` turns raw SKEMPI antibody-antigen (AB/AG) rows into the model-ready `MutationRecord` dataset: filter to AB/AG rows, build records, discard homology-duplicate siblings, assign train/val splits under two independent schemes, and write `data/processed/mutation_records.jsonl` plus the four per-split CSVs (`shared.constants.SPLIT_FILES`). See `docs/02_data_preprocessing.md` for the full design.

In [1]:
import sys
from pathlib import Path

import pandas as pd


def resolve_repo_root() -> Path:
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def add_repo_root_to_sys_path() -> Path:
    repo_root = resolve_repo_root()
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    return repo_root


REPO_ROOT = add_repo_root_to_sys_path()

## Pipeline stage counts

`data.pipeline.PipelineReport` (returned in-memory by `run_preprocessing_pipeline`, not persisted to disk) is this pipeline's stage-by-stage counter. The raw-row and homology-discard counts below are therefore cited from the last documented run (`docs/future_work.md`) rather than re-run here (re-running requires a fresh SKEMPI/PDB download); every other count is computed live from the processed files already on disk.

- Raw antibody-antigen rows in: **1211**
- Homology-duplicate samples discarded (`data.homology_dedup.discard_homology_siblings`, one representative kept per group): **154**
- Final sample count: computed below, should be `1211 - 154`.

In [2]:
from data.processed_io import load_mutation_records
from shared.constants import SPLIT_FILES, SplitName

records = load_mutation_records()
print(f"Final sample count: {len(records)}")

label_type_counts = pd.Series([record.label_type.value for record in records]).value_counts()
print("\nLabel type breakdown:")
print(label_type_counts)

Final sample count: 1057

Label type breakdown:
bounded    919
n.b.        79
ineq        59
Name: count, dtype: int64


In [3]:
def split_subset_counts() -> pd.DataFrame:
    rows = [
        {"split_scheme": split.value, "subset": subset, "n": len(pd.read_csv(SPLIT_FILES[split][subset]))}
        for split in SplitName
        for subset in ("train", "val")
    ]
    return pd.DataFrame(rows).pivot(index="split_scheme", columns="subset", values="n")


split_subset_counts()

subset,train,val
split_scheme,,
held_out_pdb,872,185
same_pdb_allowed,855,202


## Embedding backend: why SaProt alone

Three embedding backends were tried (ESM-2 sequence, IgFold structure, SaProt structure-aware PLM) — see `docs/02_data_preprocessing.md` for the full comparison, not re-derived here. **SaProt embeddings alone** (`embedding_source_mode: structure_only`) is the final decision: concatenating ESM-2 sequence embeddings alongside SaProt (`structure_and_sequence`) was tried and dropped, since at this sample count (~600-900 training rows) the larger concatenated input was too big for a light head to learn well and gave no measurable improvement over `structure_only`.

## Embedding-extraction timing

Not re-run here (heavy model calls) — these are static facts from `notebooks/saprot_extraction_probe.py` (3-PDB timing probe) and `notebooks/saprot_full_extraction_run.py` (full 1211-sample run), as documented in `docs/future_work.md`:

- **foldseek** (3Di structure tokens from the already-cached real WT PDB, all chains in one call): ~0.01-0.04s per PDB.
- **SaProt forward pass**: ~0.04-0.07s per chain (PDBs here average ~5 chains of ~100-220 residues).
- Both are negligible next to **IgFold**'s ~0.5-2s per call, and worlds away from the ESMFold OOM/~73-min-per-sample history that motivated dropping ESMFold for the antigen side entirely.
- Full extraction across all 1211 samples / 55 PDBs completed successfully with this per-unit cost, confirming SaProt is a cheap, low-risk structure backend at this dataset's scale.